# 🏆 IFest DAC — Binary Text Classification / NLI Baseline v2

## Problem Definition
Kompetisi ini bertujuan untuk mengklasifikasikan kesesuaian antara **Judul (`title`)** dan **Isi Berita (`content`)**:
- **Label 1 (Sesuai)**: Judul didukung penuh oleh isi artikel.
- **Label 0 (Tidak Sesuai)**: Judul mengalami mutasi entitas, negasi terbalik, atau tidak didukung oleh isi artikel.
- **Metrik Evaluasi**: **Macro F1-Score** $\frac{F_{1, \text{class 0}} + F_{1, \text{class 1}}}{2}$

### Aturan Kritis
- **No Pretrained Models**: Dilarang menggunakan model berbobot pretrained (IndoBERT, Transformer, dsb.). Seluruh fitur diekstraksi dan dilatih *from scratch* murni dari dataset kompetisi.

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import xgboost as xgb

print("All libraries imported successfully!")

All libraries imported successfully!


## 1. Load Data & Eksplorasi Distribusi
Dataset terdiri dari 14.397 data latih (`train.csv`) dan 3.603 data uji (`test.csv`).

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")

print("\n--- Distribusi Label Train ---")
print(train['label'].value_counts())
print(f"Rasio Class 1: {(train['label']==1).mean()*100:.2f}%")
print(f"Rasio Class 0: {(train['label']==0).mean()*100:.2f}%")

Train shape: (14397, 4)
Test shape:  (3603, 3)

--- Distribusi Label Train ---
label
1    12960
0     1437
Name: count, dtype: int64
Rasio Class 1: 90.02%
Rasio Class 0: 9.98%


## 2. Kamus Leksikal & Entitas Khusus
Bedah data menunjukkan bahwa sampel Kelas 0 dibuat melalui **mutasi entitas halus (Semantic Perturbation)**: menukar nama lokasi, negara, subjek, membalikkan negasi, atau antonim.

In [3]:
LOCATIONS = {
    'jakarta', 'jabar', 'jateng', 'jatim', 'bali', 'aceh', 'sumut', 'sumbar', 'riau',
    'surabaya', 'bandung', 'semarang', 'medan', 'makassar', 'palembang', 'bogor', 'depok',
    'tangerang', 'bekasi', 'solo', 'jogja', 'yogyakarta', 'malang', 'banyumas', 'lamongan',
    'pasuruan', 'cianjur', 'sukabumi', 'buton', 'kendari', 'tasikmalaya', 'padang',
    'amerika', 'inggris', 'jerman', 'prancis', 'china', 'cina', 'jepang', 'korsel',
    'brasil', 'brazil', 'meksiko', 'italia', 'spanyol', 'rusia', 'afsel', 'afrika',
    'australia', 'singapura', 'malaysia', 'filipina', 'thailand', 'vietnam', 'india',
    'papua', 'sulsel', 'kaltim', 'kalbar', 'kalsel', 'kaltara', 'ntb', 'ntt', 'maluku',
    'banten', 'lampung', 'bengkulu', 'jambi', 'sulut', 'sulteng', 'sultra', 'gorontalo'
}

NEGATION_WORDS = {
    'tak', 'tidak', 'bukan', 'belum', 'tanpa', 'gagal', 'batal', 'larang', 'tolak',
    'bebas', 'nihil', 'nol', 'stop', 'hentikan', 'anti', 'menolak', 'membantah',
    'menyanggah', 'menafikan', 'mustahil'
}

ANTONYM_PAIRS = [
    ('naik', 'turun'), ('menang', 'kalah'), ('positif', 'negatif'),
    ('gratis', 'bayar'), ('bebas', 'ditahan'), ('buka', 'tutup'),
    ('disiplin', 'tak disiplin'), ('tertib', 'tak tertib')
]

STOP_WORDS = {
    'dan', 'di', 'ke', 'dari', 'pada', 'untuk', 'yang', 'ini', 'itu', 'ada',
    'kini', 'lagi', 'juga', 'dalam', 'soal', 'oleh', 'atas', 'karena', 'jadi',
    'usai', 'saat', 'per', 'hingga', 'masih', 'sudah', 'telah', 'agar', 'bila',
    'jika', 'sebut', 'kata', 'ungkap', 'minta', 'terkait', 'imbau', 'bisa', 'akan'
}

## 3. Ekstraksi TF-IDF Word & Char N-Gram Cosine Similarities
Mengukur kemiripan global antara judul dengan konten penuh serta kalimat pembuka (*lead sentence*).

In [4]:
train_titles = train['title'].fillna('').astype(str).tolist()
train_contents = train['content'].fillna('').astype(str).tolist()
train_fs = [" ".join(c.split()[:35]) for c in train_contents]

test_titles = test['title'].fillna('').astype(str).tolist()
test_contents = test['content'].fillna('').astype(str).tolist()
test_fs = [" ".join(c.split()[:35]) for c in test_contents]

all_titles = train_titles + test_titles
all_contents = train_contents + test_contents
all_fs = train_fs + test_fs

print("Fitting TF-IDF Word Vectorizer...")
tfidf_word = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=15000, sublinear_tf=True)
tfidf_word.fit(all_titles + all_contents)

t_wvec_tr = tfidf_word.transform(train_titles)
c_wvec_tr = tfidf_word.transform(train_contents)
fs_wvec_tr = tfidf_word.transform(train_fs)
cos_w_c_tr = np.asarray((t_wvec_tr.multiply(c_wvec_tr)).sum(axis=1)).flatten()
cos_w_fs_tr = np.asarray((t_wvec_tr.multiply(fs_wvec_tr)).sum(axis=1)).flatten()

t_wvec_te = tfidf_word.transform(test_titles)
c_wvec_te = tfidf_word.transform(test_contents)
fs_wvec_te = tfidf_word.transform(test_fs)
cos_w_c_te = np.asarray((t_wvec_te.multiply(c_wvec_te)).sum(axis=1)).flatten()
cos_w_fs_te = np.asarray((t_wvec_te.multiply(fs_wvec_te)).sum(axis=1)).flatten()

print("Fitting TF-IDF Char-wb Vectorizer...")
tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=5, max_features=12000, sublinear_tf=True)
tfidf_char.fit(all_titles + all_fs)

t_cvec_tr = tfidf_char.transform(train_titles)
c_cvec_tr = tfidf_char.transform(train_contents)
fs_cvec_tr = tfidf_char.transform(train_fs)
cos_c_c_tr = np.asarray((t_cvec_tr.multiply(c_cvec_tr)).sum(axis=1)).flatten()
cos_c_fs_tr = np.asarray((t_cvec_tr.multiply(fs_cvec_tr)).sum(axis=1)).flatten()

t_cvec_te = tfidf_char.transform(test_titles)
c_cvec_te = tfidf_char.transform(test_contents)
fs_cvec_te = tfidf_char.transform(test_fs)
cos_c_c_te = np.asarray((t_cvec_te.multiply(c_cvec_te)).sum(axis=1)).flatten()
cos_c_fs_te = np.asarray((t_cvec_te.multiply(fs_cvec_te)).sum(axis=1)).flatten()

print("TF-IDF features computed successfully!")

Fitting TF-IDF Word Vectorizer...
Fitting TF-IDF Char-wb Vectorizer...
TF-IDF features computed successfully!


## 4. Feature Engineering: Entity Discrepancy & Sentence-Level Window Alignment
Fungsi ini mengekstrak 30 fitur NLI komprehensif, mencakup:
1. **Entity Absence Check**: Mengecek apakah entitas kapital / proper noun di judul hilang dari konten.
2. **Best Window Matching**: Menggeser jendela 25 kata pada konten untuk mencari kalimat yang paling mirip judul, lalu mengukur kata-kata judul yang tidak ada di kalimat terbaik tersebut.
3. **Interaction Feature**: `high_match_missing_entity` (judul sangat mirip dengan kalimat terbaik, tetapi ada 1 entitas nama yang hilang sama sekali).
4. **Location, Negation, Antonym, Number Mismatches**.

In [5]:
def extract_features(titles_list, contents_list, first_sents_list, cos_w_c, cos_w_fs, cos_c_c, cos_c_fs):
    feats = []
    for i in range(len(titles_list)):
        t_raw = titles_list[i]
        c_raw = contents_list[i]
        
        t_words = t_raw.split()
        t_lower = t_raw.lower()
        c_lower = c_raw.lower()
        fs_lower = first_sents_list[i].lower()
        
        t_tokens = set(re.findall(r'\w+', t_lower))
        c_tokens = set(re.findall(r'\w+', c_lower))
        fs_tokens = set(re.findall(r'\w+', fs_lower))
        
        # 1. Locations
        t_locs = {w for w in t_tokens if w in LOCATIONS}
        loc_mismatch = 0.0
        if t_locs:
            missing_locs = [loc for loc in t_locs if loc not in c_tokens]
            loc_mismatch = len(missing_locs) / len(t_locs)
            
        # 2. Negations & Antonyms
        t_negs = t_tokens.intersection(NEGATION_WORDS)
        fs_negs = fs_tokens.intersection(NEGATION_WORDS)
        c_negs = c_tokens.intersection(NEGATION_WORDS)
        neg_conflict_fs = 1.0 if (bool(t_negs) != bool(fs_negs)) else 0.0
        neg_conflict_doc = 1.0 if (bool(t_negs) != bool(c_negs)) else 0.0
        
        ada_conflict = 0.0
        if ('ada' in t_tokens and not t_negs):
            if 'tidak ada' in fs_lower or 'tak ada' in fs_lower:
                ada_conflict = 1.0
        elif ('tidak ada' in t_lower or 'tak ada' in t_lower):
            if 'ada' in fs_tokens and 'tidak ada' not in fs_lower and 'tak ada' not in fs_lower:
                ada_conflict = 1.0
                
        antonym_conflict = 0.0
        for w1, w2 in ANTONYM_PAIRS:
            if (w1 in t_tokens and w2 in c_tokens) or (w2 in t_tokens and w1 in c_tokens):
                antonym_conflict = 1.0
                break
                
        # 3. Capitalized Entities / Proper Nouns
        cap_words = [w for w in t_words if len(w) > 1 and w[0].isupper() and w.lower() not in STOP_WORDS]
        cap_missing = sum(1 for w in cap_words if w.lower() not in c_tokens)
        cap_missing_fs = sum(1 for w in cap_words if w.lower() not in fs_tokens)
        cap_cov_c = 1.0 - (cap_missing / (len(cap_words) + 1e-9)) if cap_words else 1.0
        cap_cov_fs = 1.0 - (cap_missing_fs / (len(cap_words) + 1e-9)) if cap_words else 1.0
        
        cap_bigrams = []
        for j in range(len(t_words) - 1):
            w1_clean = re.sub(r'[^a-zA-Z]', '', t_words[j])
            w2_clean = re.sub(r'[^a-zA-Z]', '', t_words[j+1])
            if len(w1_clean) > 1 and len(w2_clean) > 1 and w1_clean[0].isupper() and w2_clean[0].isupper():
                if w1_clean.lower() not in STOP_WORDS and w2_clean.lower() not in STOP_WORDS:
                    cap_bigrams.append(f"{w1_clean.lower()} {w2_clean.lower()}")
                    
        missing_bigrams = sum(1 for bg in cap_bigrams if bg not in c_lower)
        ratio_missing_bigrams = missing_bigrams / (len(cap_bigrams) + 1e-9) if cap_bigrams else 0.0
        
        # 4. Numbers
        t_nums = set(re.findall(r'\d+', t_raw))
        c_nums = set(re.findall(r'\d+', c_raw))
        num_missing = len(t_nums - c_nums)
        num_cov = len(t_nums.intersection(c_nums)) / (len(t_nums) + 1e-9) if t_nums else 1.0
        
        # 5. Token Coverage & Bigrams
        useful_tokens = [w for w in t_tokens if w not in STOP_WORDS]
        tok_cov_c = len([w for w in useful_tokens if w in c_tokens]) / (len(useful_tokens) + 1e-9)
        tok_cov_fs = len([w for w in useful_tokens if w in fs_tokens]) / (len(useful_tokens) + 1e-9)
        
        t_content_words = [w.lower() for w in t_words if w.lower() not in STOP_WORDS and len(w) > 2]
        t_bigrams = [f"{t_content_words[j]} {t_content_words[j+1]}" for j in range(len(t_content_words)-1)]
        bg_cov_c = sum(1 for bg in t_bigrams if bg in c_lower) / (len(t_bigrams) + 1e-9) if t_bigrams else 1.0
        bg_cov_fs = sum(1 for bg in t_bigrams if bg in fs_lower) / (len(t_bigrams) + 1e-9) if t_bigrams else 1.0
        
        # 6. Best Window Alignment (Sentence level)
        c_words = c_raw.split()
        windows = []
        w_size = 25
        step = 12
        for start in range(0, len(c_words), step):
            chunk = " ".join(c_words[start:start+w_size]).lower()
            if len(chunk.split()) >= 5:
                windows.append(chunk)
        if not windows:
            windows = [c_lower]
            
        best_overlap = 0
        best_window = windows[0]
        best_win_idx = 0
        for w_idx, win in enumerate(windows):
            w_toks = set(re.findall(r'\w+', win))
            overlap = len([w for w in useful_tokens if w in w_toks])
            if overlap > best_overlap:
                best_overlap = overlap
                best_window = win
                best_win_idx = w_idx
                
        best_w_tokens = set(re.findall(r'\w+', best_window))
        best_win_cov = best_overlap / (len(useful_tokens) + 1e-9)
        jaccard_best_win = best_overlap / (len(set(useful_tokens).union(best_w_tokens)) + 1e-9)
        
        # 7. Interaction Feature: High Overlap BUT Missing Entity
        high_match_missing_entity = 1.0 if (best_win_cov >= 0.5 and cap_missing > 0) else 0.0
        high_match_missing_bigram = 1.0 if (best_win_cov >= 0.5 and missing_bigrams > 0) else 0.0
        
        len_ratio = len(c_raw) / (len(t_raw) + 1e-9)
        
        feats.append([
            cos_w_c[i],
            cos_w_fs[i],
            cos_c_c[i],
            cos_c_fs[i],
            loc_mismatch,
            neg_conflict_fs,
            neg_conflict_doc,
            ada_conflict,
            antonym_conflict,
            cap_missing,
            cap_missing_fs,
            cap_cov_c,
            cap_cov_fs,
            missing_bigrams,
            ratio_missing_bigrams,
            num_missing,
            num_cov,
            tok_cov_c,
            tok_cov_fs,
            bg_cov_c,
            bg_cov_fs,
            best_win_cov,
            jaccard_best_win,
            best_win_idx,
            high_match_missing_entity,
            high_match_missing_bigram,
            len_ratio,
            len(cap_words),
            len(t_words),
            len(c_words)
        ])
    return np.array(feats)

print("Extracting feature matrices...")
X_train = extract_features(train_titles, train_contents, train_fs, cos_w_c_tr, cos_w_fs_tr, cos_c_c_tr, cos_c_fs_tr)
X_test = extract_features(test_titles, test_contents, test_fs, cos_w_c_te, cos_w_fs_te, cos_c_c_te, cos_c_fs_te)
y_train = train['label'].values

print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")

Extracting feature matrices...
X_train shape: (14397, 30) | X_test shape: (3603, 30)


## 5. Stratified 5-Fold Cross Validation & Multi-Model Ensemble
Kita menerapkan:
- **Class Weighting (5.0x)** pada Kelas 0 untuk menyeimbangkan penalti terhadap false positives.
- **Ensemble Blending**: Menggabungkan probabilitas `0.4 * XGBoost + 0.4 * HistGradientBoosting + 0.2 * ExtraTrees`.
- **Threshold Optimization**: Melakukan sweep threshold presisi 0.005 pada probabilitas Out-Of-Fold (OOF).

In [6]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
sample_weights = np.where(y_train == 0, 5.0, 1.0)

xgb_oof = np.zeros(len(train))
hgb_oof = np.zeros(len(train))
et_oof = np.zeros(len(train))

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"--- Training Fold {fold+1} ---")
    
    # Model 1: XGBoost
    clf_xgb = xgb.XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.035,
        subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='logloss'
    )
    clf_xgb.fit(X_train[trn_idx], y_train[trn_idx], sample_weight=sample_weights[trn_idx])
    xgb_oof[val_idx] = clf_xgb.predict_proba(X_train[val_idx])[:, 1]
    
    # Model 2: HistGradientBoosting
    clf_hgb = HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.035, max_depth=6, random_state=42
    )
    clf_hgb.fit(X_train[trn_idx], y_train[trn_idx], sample_weight=sample_weights[trn_idx])
    hgb_oof[val_idx] = clf_hgb.predict_proba(X_train[val_idx])[:, 1]
    
    # Model 3: ExtraTrees
    clf_et = ExtraTreesClassifier(n_estimators=250, max_depth=12, random_state=42, n_jobs=-1)
    clf_et.fit(X_train[trn_idx], y_train[trn_idx], sample_weight=sample_weights[trn_idx])
    et_oof[val_idx] = clf_et.predict_proba(X_train[val_idx])[:, 1]

# Ensemble OOF Probabilities
ens_oof = 0.4 * xgb_oof + 0.4 * hgb_oof + 0.2 * et_oof

# Find Best Threshold for Macro F1
best_th, best_f1 = 0.5, 0.0
for th in np.arange(0.1, 0.95, 0.005):
    preds = (ens_oof >= th).astype(int)
    sc = f1_score(y_train, preds, average='macro')
    if sc > best_f1:
        best_f1 = sc
        best_th = th

print("\n" + "="*50)
print(f"OPTIMAL THRESHOLD: {best_th:.3f} | MACRO F1: {best_f1:.4f}")
print("="*50)
oof_preds = (ens_oof >= best_th).astype(int)
print(classification_report(y_train, oof_preds, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_train, oof_preds))

--- Training Fold 1 ---
--- Training Fold 2 ---
--- Training Fold 3 ---
--- Training Fold 4 ---
--- Training Fold 5 ---

OPTIMAL THRESHOLD: 0.465 | MACRO F1: 0.6579
              precision    recall  f1-score   support

           0     0.4651    0.3111    0.3728      1437
           1     0.9263    0.9603    0.9430     12960

    accuracy                         0.8955     14397
   macro avg     0.6957    0.6357    0.6579     14397
weighted avg     0.8803    0.8955    0.8861     14397

Confusion Matrix:
[[  447   990]
 [  514 12446]]


## 6. Prediksi Data Uji (`test.csv`) & Pembuatan Submission
Melatih model pada seluruh data latih, memprediksi probabilitas data uji, menerapkan threshold optimal, dan menyimpan ke file CSV.

In [7]:
print("Training full models on all 14,397 training rows...")

clf_xgb_full = xgb.XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.035,
    subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='logloss'
)
clf_xgb_full.fit(X_train, y_train, sample_weight=sample_weights)
p_xgb_test = clf_xgb_full.predict_proba(X_test)[:, 1]

clf_hgb_full = HistGradientBoostingClassifier(
    max_iter=300, learning_rate=0.035, max_depth=6, random_state=42
)
clf_hgb_full.fit(X_train, y_train, sample_weight=sample_weights)
p_hgb_test = clf_hgb_full.predict_proba(X_test)[:, 1]

clf_et_full = ExtraTreesClassifier(n_estimators=250, max_depth=12, random_state=42, n_jobs=-1)
clf_et_full.fit(X_train, y_train, sample_weight=sample_weights)
p_et_test = clf_et_full.predict_proba(X_test)[:, 1]

# Final Test Probabilities
p_test = 0.4 * p_xgb_test + 0.4 * p_hgb_test + 0.2 * p_et_test
test_preds = (p_test >= best_th).astype(int)

submission = pd.DataFrame({'id': test['id'], 'label': test_preds})
print("\n--- Distribusi Prediksi Submission ---")
print(submission['label'].value_counts())
print(f"Persentase Kelas 1: {(submission['label']==1).mean()*100:.2f}%")
print(f"Persentase Kelas 0: {(submission['label']==0).mean()*100:.2f}%")

submission.to_csv('submission_step1_ensemble.csv', index=False)
submission.to_csv('submission_baseline.csv', index=False)
print("\nSubmission files saved successfully: 'submission_baseline.csv' and 'submission_step1_ensemble.csv'!")

Training full models on all 14,397 training rows...

--- Distribusi Prediksi Submission ---
label
1    3372
0     231
Name: count, dtype: int64
Persentase Kelas 1: 93.59%
Persentase Kelas 0: 6.41%

Submission files saved successfully: 'submission_baseline.csv' and 'submission_step1_ensemble.csv'!
